# Adopt the previous results — exp02 and exp03 only

A one-off. Run it once, then go back to `10_dqn_suite_runner.ipynb` and never
think about it again.

## What this does, and what it deliberately does not

The exploratory notebooks wrote to `tfm_qrl/exp01`, `tfm_qrl/exp02`, … The suite
expects one root with a directory named after each experiment script. Until the
folders are renamed, the suite looks in an empty directory, finds nothing, and
recomputes everything you already have.

Two operations, often confused because they run back to back:

| script | what it does |
|---|---|
| `adopt_legacy_layout.py` | **moves** folders into the layout the suite reads |
| `migrate_manifests.py` | **back-fills fields inside** the manifests so the reuse guard can verify them |

Both are needed. Moving alone leaves manifests the guard cannot check; migrating
alone leaves them somewhere the suite never looks.

## exp01 is left where it is, on purpose

exp01 ran at **60,000 steps**; exp02 and exp03 ran at 100,000. Adopting exp01 as
it stands would either force the whole suite down to a 60k budget, or trip the
reuse guard on every exp01 cell — correctly, because a 60k run is not an answer
to a 100k question.

There is a better reason to re-run it than mere tidiness. exp03's step budget was
raised from 60k to 100k precisely because *"DR needed more budget to unfold"* —
which is direct evidence, from this project, that 60k was too short for the
hybrid arms to separate. exp01's negative result (no clear circuit advantage at
equal budget and information) was measured inside that window. It may well
survive at 100k, and if it does it becomes considerably harder to argue with.
If it does not survive, that is something worth knowing before the memoria is
written rather than after.

So this notebook leaves `tfm_qrl/exp01` untouched. The runs stay on Drive as a
record, and exp01 is re-run at 100k into the new layout, where it sits alongside
exp02, exp03 and exp03b on the same budget.

---
## 1. Setup

In [ ]:
import os, sys, subprocess, pathlib

GITHUB_USER, REPO_NAME, BRANCH = "RogerMas99", "qrl-dissection", "main"
try:
    from google.colab import userdata
    _tok = userdata.get("GH_TOKEN")
    REPO_URL = (f"https://{_tok}@github.com/{GITHUB_USER}/{REPO_NAME}.git" if _tok
                else f"https://github.com/{GITHUB_USER}/{REPO_NAME}.git")
except Exception:
    REPO_URL = f"https://github.com/{GITHUB_USER}/{REPO_NAME}.git"

IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    from google.colab import drive; drive.mount("/content/drive")
    CODE = pathlib.Path("/content/qrl-dissection")
    DRIVE = pathlib.Path("/content/drive/MyDrive/tfm_qrl")
else:
    CODE = pathlib.Path.cwd().parent if pathlib.Path.cwd().name == "notebooks" else pathlib.Path.cwd()
    DRIVE = CODE / "results_drive_sim"
RES = DRIVE / "results"

if IN_COLAB:
    if CODE.exists():
        subprocess.run(["git", "-C", str(CODE), "pull", "--quiet"], check=False)
    else:
        subprocess.run(["git", "clone", "--quiet", "-b", BRANCH, REPO_URL, str(CODE)], check=True)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-e", str(CODE)], check=True)

sys.path.insert(0, str(CODE / "src"))
print(f"code   {CODE}")
print(f"drive  {DRIVE}")
print(f"target {RES}")

---
## 2. Before

Read this carefully before changing anything. In particular check the `steps`
line of each experiment — that is what determines whether an existing run can be
reused at all.

In [ ]:
!cd {CODE} && python scripts/inventory_results.py {DRIVE} -v

---
## 3. Move exp02 and exp03

`--only` names exactly which folders to adopt. Anything left out is reported and
left alone. Dry run first.

In [ ]:
!cd {CODE} && python scripts/adopt_legacy_layout.py {DRIVE} --only exp02 exp03 --dry-run

If the plan above looks right, run it for real. Folders are **moved**, not copied, and nothing is deleted — if a target already exists the source is left alone and reported.

In [ ]:
!cd {CODE} && python scripts/adopt_legacy_layout.py {DRIVE} --only exp02 exp03

---
## 4. Back-fill the manifests

One command per experiment, because `--dqn-kwargs` differs between them and
cannot be recovered from any file. The values below are read out of the
experiment scripts themselves — `experiments/exp02_...py` and
`experiments/exp03_...py`, both `kw = dict(...)`. Do not type them from memory:
whatever you write becomes the value every future run is checked against.

Recovered automatically: the step budget (from the outcome), the seed and the
FIX-01 state from the run name. Backups are written as `*.manifest.json.bak`.

In [ ]:
!cd {CODE} && python scripts/migrate_manifests.py {RES}/exp02_dqn_cartpole_output_reuse \
    --arm hybrid_fig4 \
    --dqn-kwargs '{{"batch_size":128,"buffer_size":10000,"train_frequency":10}}' --dry-run

!cd {CODE} && python scripts/migrate_manifests.py {RES}/exp03_dqn_cartpole_data_reuploading \
    --arm hybrid_fig4 \
    --dqn-kwargs '{{"batch_size":128,"buffer_size":10000,"train_frequency":10}}' --dry-run

Same two commands without `--dry-run`:

In [ ]:
!cd {CODE} && python scripts/migrate_manifests.py {RES}/exp02_dqn_cartpole_output_reuse \
    --arm hybrid_fig4 \
    --dqn-kwargs '{{"batch_size":128,"buffer_size":10000,"train_frequency":10}}'

!cd {CODE} && python scripts/migrate_manifests.py {RES}/exp03_dqn_cartpole_data_reuploading \
    --arm hybrid_fig4 \
    --dqn-kwargs '{{"batch_size":128,"buffer_size":10000,"train_frequency":10}}'

---
## 5. Verify

Two checks. The first should report **0 legacy manifests**. The second should
show exp02 and exp03 as largely done, and exp01 as pending — that is the intended
outcome, not a problem.

In [ ]:
!cd {CODE} && python scripts/inventory_results.py {RES} -v

In [ ]:
!cd {CODE} && python scripts/run_dqn_suite.py --plan --pass coverage --outroot {RES}

### Reading the plan

`exp02` and `exp03` should show their existing cells as done. If they still show
`0`, one of two things went wrong: the move did not happen (check step 3's
output), or the manifests are still legacy (check step 5's first cell). Neither
is destructive — the runs are on Drive either way.

`exp01` showing `0 / 18` is correct. Its 60k runs are still in `tfm_qrl/exp01`,
untouched.

---
## 6. exp01 at 100k

Nothing forces this now, but it is the natural next step: it puts every CartPole
experiment on the same 100k budget and re-tests the negative result inside a
window wide enough for the hybrid arm to unfold.

`--max-cells` rather than a wall clock, because the hybrid cells run for hours
and a time limit cannot preempt one — it would let the runtime disconnect kill a
half-trained cell instead.

In [ ]:
!cd {CODE} && python scripts/run_dqn_suite.py --only exp01 --pass coverage \
    --outroot {RES} --steps 100000 --max-cells 2 --skip-preflight

Rerun that cell until `--plan` shows exp01 complete. Finished cells are skipped,
so each session picks up where the last one stopped.

**When it finishes, update `docs/RESULTS-LOG.md`.** exp01's entry currently
records a 60k result; if the 100k numbers differ, the old entry is superseded
rather than annotated, and the 60k runs remain in `tfm_qrl/exp01` as the record
of what was measured first.

Then go back to `10_dqn_suite_runner.ipynb`.